# Use Case — Public-Parks Heat-Resilience Audit

**Who this is for**  
City parks-and-recreation directors, public-health environmental health teams, and climate-resilience officers. The persona is the same person who would commission a tree-canopy survey or a playground shade audit — but here the input is a **parks GIS export** and the output is a per-park diagnosis tied to nationally-funded improvement programs.

**The scenario**  
Summer is here. Park usage drops on hot days, which has measurable public-health consequences: less physical activity (a CDC-tracked outcome), and for the people who *do* go, real heat-illness risk. Parks in neighborhoods where homes lack AC are *de facto* cooling refuges — when those parks are hot, residents have nowhere to go. You manage a list of parks; you need to know which ones are exposing the public to dangerous heat, **why** each is hot, and **what** specific improvement to fund — without inventing a score and without quoting a dollar figure that doesn't generalize from one city to another.

This notebook combines **your park list** with **FortyGuard layers** to answer four public-health questions:

1. **Where is it hot, when, and by how much?**  ← 24-hour heatmap × your park points
2. **Why is each park hot?**  ← satellite segmentation on the top exposures (canopy %, impervious %)
3. **What does it look like at ground level where the public actually stands?**  ← street view at the worst park
4. **Is the heat a humidity problem (heat-index) or a dry-heat problem during use hours?**  ← env-params at the top exposures

**The output is declarative, not synthetic.** Every column you see is a direct API measurement (temperature, heat index, canopy %, impervious %, sky %). Every recommendation is a *threshold trigger* — *if measurement X crosses a published NOAA / EPA / CDC / USDA / NRPA threshold, then recommend the program that funds the fix*. No invented index. No dollar value. Portable to any city in the country.

> **Cached by default.** The notebook ships with `CACHED=True` so it runs end-to-end against the bundled San Jose sample files in `data/`. Set `CACHED=False` (and add `FORTYGUARD_API_KEY` to `.env`) to run live against any AOI.

> **Bring your own data.** Sample park list ships at `data/sample_public_parks.csv`. Swap the path in Step 1 — as long as the columns match (`park_id`, `name`, `type`, `acres`, `latitude`, `longitude`), everything downstream works.

**Why this is different from the existing facility-cooling notebook**: that one weights facility points by *vulnerable_population* (a number you have to supply per facility). This one needs only what every parks office already publishes — a list of parks with coordinates — and explains heat in terms of the *physical environment* the API directly measures.

---

## Setup

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import json
import numpy as np
import pandas as pd
import folium
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import Point, shape, mapping
from IPython.display import HTML, display

from fortyguard import FortyGuardClient
from fortyguard.exceptions import FortyGuardError

# ── configuration ─────────────────────────────────────────
STUDY_DATE        = '2024-10-02'         # matches the bundled cached files
STUDY_HOUR        = '00:00'              # design-peak snapshot for live API mode
GRANULARITY_M     = 80                  # heatmap resolution
CACHED            = True                 # set False for live FortyGuard calls
TOP_N_TO_ENRICH   = 3                    # depth-of-diagnosis budget (satellite + env-params)
USE_HOUR_START    = 10                   # daytime use window for heat-index thresholding
USE_HOUR_END      = 18                   # 18:00 inclusive (covers afternoon + early-evening peak)
TEMP_C_SANITY     = (15.0, 55.0)         # F→C sanity range for the cached heatmap

# ── published thresholds (no invented numbers — every value below comes from a public source) ─
# Heat Index categories: NOAA NWS Heat Index Chart
HI_CAUTION_C          = 27.0   # NOAA 'Caution' boundary (heat index ≥ 27 °C / 80 °F)
HI_EXTREME_CAUTION_C  = 32.0   # NOAA 'Extreme Caution' boundary (heat index ≥ 32 °C / 90 °F)
HI_DANGER_C           = 39.0   # NOAA 'Danger' boundary (heat index ≥ 39 °C / 103 °F)
# Surface composition: EPA Heat Island Reduction guidance + USDA i-Tree program targets
CANOPY_TARGET_PCT     = 25.0   # EPA / USDA target tree canopy in urban park settings
IMPERVIOUS_TRIGGER    = 60.0   # EPA Heat Island guidance: > 60% impervious is a cool-pavement candidate
# Ground-level: NRPA shade-equity guidance (parks with high sky-fraction at play areas need shade structures)
SKY_TRIGGER_PCT       = 60.0   # NRPA: sky fraction > 60% in front-view of play area = unshaded
# Activity guidance: CDC physical-activity + heat guidance
RH_HIGH_PCT           = 60.0   # > 60% RH at peak combined with > 30 °C ambient triggers splash-pad / mister review

# ── published-program citation strings (used in the action brief, no values invented) ────────
PROG_USDA_ITREE       = 'USDA Forest Service i-Tree · EPA Heat Island Reduction (Trees & Vegetation)'
PROG_EPA_HEATISLAND   = 'EPA Heat Island Reduction (Cool Pavements)'
PROG_NRPA_SHADE       = 'NRPA Shade-Equity guidance · CDC BRACE shade-structure programs'
PROG_CDC_BRACE        = 'CDC BRACE — heat advisory + activity-window guidance'
PROG_CDC_HEAT_RES     = 'CDC heat-resilience programs (splash pads / misters)'

# ── data paths ────────────────────────────────────────────
# Cached files are San Jose samples bundled with the repo. They are at a real
# city-block location and stand in for the worst-park deep-dive in CACHED mode.
# Set CACHED=False to refetch at the actual park coordinates.
DATA            = ROOT / 'data'
PARKS_CSV       = DATA / 'sample_public_parks.csv'
HEATMAP_GEOJSON = DATA / 'real_state_san_jose_heatmap_sample_day_2024-10-02.geojson'
ENV_PARAMS_JSON = DATA / 'real_state_san_jose_env_paramaters_sample_day_2024-10-02.json'
SATELLITE_JSON  = DATA / 'real_state_san_jose_satellite_segmentation_sample_day_2024-10-02.json'
STREETVIEW_JSON = DATA / 'real_state_san_jose_street_view_segmentation_sample_day_2024-10-02.json'
OUTPUT_CSV      = ROOT / 'outputs' / 'public_parks_heat_audit.csv'
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Live client — only used when CACHED=False.
try:
    client = FortyGuardClient()
    HAVE_API = True
except FortyGuardError as exc:
    client = None
    HAVE_API = False
    print(f'[no API key] running in offline mode: {exc}')

# ── shared color ramp ────────────────────────────────────────
TCM_COLORS = [
    '#2983ba', '#5aa4b2', '#88c4aa', '#b3e0a6', '#d1ecb0', '#f0f9ba',
    '#fff0ae', '#fed38c', '#fdb56a', '#f3854e', '#e54f35', '#d7191c',
]
TCM_CMAP = LinearSegmentedColormap.from_list('tcm', TCM_COLORS, N=256)
def temp_color(t, lo, hi):
    if t is None or hi == lo: return TCM_COLORS[0]
    frac = max(0.0, min(1.0, (float(t) - lo) / (hi - lo)))
    return TCM_COLORS[min(int(frac * len(TCM_COLORS)), len(TCM_COLORS) - 1)]

PARK_TYPE_PALETTE = {
    'Regional'    : '#1f77b4',
    'Neighborhood': '#2ca02c',
    'Plaza'       : '#9467bd',
    'Sports'      : '#ff7f0e',
    'Playground'  : '#e377c2',
    'Trail-head'  : '#8c564b',
    'Dog-Park'    : '#7f7f7f',
}

print(f'CACHED={CACHED}  STUDY_DATE={STUDY_DATE}  TOP_N_TO_ENRICH={TOP_N_TO_ENRICH}')
print(f'Use-hour window for heat-index thresholding: {USE_HOUR_START:02d}:00 – {USE_HOUR_END:02d}:00')

---
## Step 1 — Load your park list

### What you are doing
Reading a parks point CSV. The schema is the minimum that any city's parks GIS already exports — `park_id`, `name`, `type`, `acres`, `latitude`, `longitude`. No demographic columns required, no vulnerability weights to author by hand.

### Why this matters (public health)
Starting from your own park list keeps the audit portable. Parks departments in different states maintain different fields; the only ones the heat analysis actually needs are coordinates. Everything else passes through to the final CSV so the recommendations land in *your* GIS, not in a stand-alone deliverable that has to be hand-merged later.

In [ ]:
parks = pd.read_csv(PARKS_CSV)
type_counts = parks['type'].value_counts().to_dict()
print(f"Loaded {len(parks)} parks covering {parks['acres'].sum():.0f} total acres")
print(f"  type mix: {type_counts}")
parks

---
## Step 2 — Heat layer (cached or live)

### What you are doing
When `CACHED=True` we read the bundled 24-hour heatmap GeoJSON (~16 k tiles, hourly temperatures `'00'..'23'` in °F → converted to °C with a sanity assertion). When `CACHED=False` we call `client.create_heatmap` for the design hour. Either way the result is a list of `(polygon, hourly_c, peak_c, peak_h, mean_c)` tiles.

### Why this matters (public health)
Public-health heat exposure is a *day*-shaped problem, not a single-hour snapshot. People use parks across the full afternoon, and the heat-illness risk window is the slice when heat-index is high *and* the park is in use — usually 10:00–18:00. The 24-hour cached layer gives us hourly temperatures at every park's tile, which is what we need to compute *hours-above-Caution* in the use window further down.

In [ ]:
import textwrap

def _f_to_c(f):
    return (f - 32.0) * 5.0 / 9.0

def _load_cached_heatmap():
    with open(HEATMAP_GEOJSON, 'r', encoding='utf-8') as f:
        gj = json.load(f)
    tiles = []
    minx = miny =  1e9
    maxx = maxy = -1e9
    for ft in gj.get('features', []):
        poly = shape(ft['geometry'])
        props = ft['properties']
        hourly_c = [_f_to_c(props[f'{h:02d}']) for h in range(24)]
        peak_c   = max(hourly_c)
        peak_h   = hourly_c.index(peak_c)
        mean_c   = sum(hourly_c) / 24.0
        tiles.append((poly, hourly_c, peak_c, peak_h, mean_c))
        x0, y0, x1, y1 = poly.bounds
        if x0 < minx: minx = x0
        if y0 < miny: miny = y0
        if x1 > maxx: maxx = x1
        if y1 > maxy: maxy = y1
    return tiles, (minx, miny, maxx, maxy)

def show_heatmap_summary(temps, source_label):
    """Stats card + colored histogram + vertical colorbar — one figure summarizing
    the AOI temperature distribution. `temps` is the per-tile peak (°C) list.
    Same visual as the real-estate and bus-stops notebooks."""
    temps = [t for t in temps if t is not None]
    if not temps:
        print('No tile temperatures to summarize.')
        return
    lo, hi = float(min(temps)), float(max(temps))
    mean   = float(sum(temps) / len(temps))

    wrapped_label = textwrap.fill(source_label, width=22) if source_label else ''
    n_label_lines = wrapped_label.count('\n') + 1

    fig = plt.figure(figsize=(12, 3.4 + 0.30 * max(0, n_label_lines - 1)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    # Stats card -----------------------------------------------------------
    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    ax0.text(0.0, 0.97, wrapped_label, transform=ax0.transAxes,
             fontsize=10.5, fontweight='bold', color='#222', va='top')
    subtitle_y = 0.97 - 0.11 * n_label_lines - 0.05
    ax0.text(0.0, subtitle_y, f"{len(temps):,} tiles",
             transform=ax0.transAxes,
             fontsize=10, color='#666', va='top')

    rows = [('min',  lo,   temp_color(lo,   lo, hi)),
            ('mean', mean, temp_color(mean, lo, hi)),
            ('max',  hi,   temp_color(hi,   lo, hi))]
    band_top    = subtitle_y - 0.10
    band_bottom = 0.05
    step        = (band_top - band_bottom) / max(len(rows) - 1, 1)
    rect_h      = min(0.14, step * 0.6)
    y = band_top
    for label, val, color in rows:
        ax0.text(0.0, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace')
        ax0.add_patch(plt.Rectangle((0.22, y - rect_h / 2), 0.10, rect_h,
                                    transform=ax0.transAxes,
                                    facecolor=color, edgecolor='#333', linewidth=0.6))
        ax0.text(0.37, y, f"{val:.2f} °C", transform=ax0.transAxes,
                 fontsize=13, fontweight='bold', color='#222', va='center',
                 family='monospace')
        y -= step

    # Histogram ------------------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, edge_lo, edge_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((edge_lo + edge_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for spine in ('top', 'right'):
        ax1.spines[spine].set_visible(False)

    # Colorbar -------------------------------------------------------------
    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP,
               extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([])
    ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)

    plt.show()

if CACHED:
    tiles, aoi_bounds = _load_cached_heatmap()
    peaks = [t[2] for t in tiles]
    lo, hi = min(peaks), max(peaks)
    assert TEMP_C_SANITY[0] <= lo and hi <= TEMP_C_SANITY[1], \
        f'F→C sanity check failed: peak range {lo:.1f}..{hi:.1f} outside {TEMP_C_SANITY}'
    print(f'[cached] {len(tiles):,} tiles, peak range {lo:.1f}..{hi:.1f} °C')
    show_heatmap_summary(peaks, f'Cached · {HEATMAP_GEOJSON.name}')
else:
    if not HAVE_API:
        raise RuntimeError('Live mode requested but no API key available.')
    rx0 = parks['longitude'].min() - 0.005
    rx1 = parks['longitude'].max() + 0.005
    ry0 = parks['latitude' ].min() - 0.005
    ry1 = parks['latitude' ].max() + 0.005
    aoi = {'type':'FeatureCollection','features':[{
        'type':'Feature','properties':{},
        'geometry':{'type':'Polygon','coordinates':[[
            [rx0, ry0],[rx1, ry0],[rx1, ry1],[rx0, ry1],[rx0, ry0]]]}}]}
    heatmap = client.create_heatmap(
        polygon_aoi=aoi, start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=3, granularity=GRANULARITY_M, verbose=False)
    feats = (heatmap['result'].get('map_data') or {}).get('features', [])
    sh = int(STUDY_HOUR.split(':')[0])
    tiles, minx, miny, maxx, maxy = [], 1e9, 1e9, -1e9, -1e9
    for ft in feats:
        poly = shape(ft['geometry'])
        t = ft['properties'].get('temperature')
        hourly_c = [t] * 24
        tiles.append((poly, hourly_c, t, sh, t))
        x0, y0, x1, y1 = poly.bounds
        minx, miny = min(minx, x0), min(miny, y0)
        maxx, maxy = max(maxx, x1), max(maxy, y1)
    aoi_bounds = (minx, miny, maxx, maxy)
    peaks = [t[2] for t in tiles]
    print(f'[live] {len(tiles):,} tiles, single hour {STUDY_HOUR}.')
    print('[live] ⚠ Diurnal columns collapse to the snapshot hour; use CACHED=True for full-day analysis.')
    show_heatmap_summary(peaks, f'Live API · {STUDY_DATE} {STUDY_HOUR}')

tile_means = [t[4] for t in tiles]
tile_peaks = [t[2] for t in tiles]
print(f'AOI mean tile temperature: {sum(tile_means)/len(tile_means):.2f} °C')
print(f'AOI peak tile temperature: {max(tile_peaks):.2f} °C')

def tile_for(lat, lon):
    p = Point(lon, lat)
    for t in tiles:
        if t[0].contains(p): return t
    return min(tiles, key=lambda t: t[0].centroid.distance(p))

---
## Step 3 — Diurnal temperature attach

### What you are doing
For each park, find the heatmap tile that contains its coordinates and copy off **peak temperature**, **peak hour**, **daily-mean temperature**, **diurnal swing**, **AOI percentile**, and **hours-above-Caution-during-use-window** — the count of hours in 10:00–18:00 when the tile reads ≥ 27 °C (NOAA Heat-Index Caution boundary). Now every park has an analysis-ready row driven entirely by API output.

### Why this matters (public health)
*Hours-above-Caution-during-use-window* is the column that connects the heat measurement to a public-health consequence. NOAA Caution means "fatigue possible with prolonged exposure" — every hour past that boundary is an hour during which a parent has to ask whether the playground visit is safe. A peak of 35 °C at 03:00 doesn't matter; a peak of 33 °C from 13:00 to 18:00 means the park is functionally unusable for half its operating day.

In [ ]:
def _percentile_rank(value, sorted_values):
    lo, hi = 0, len(sorted_values)
    while lo < hi:
        mid = (lo + hi) // 2
        if sorted_values[mid] <= value: lo = mid + 1
        else: hi = mid
    return round(100.0 * lo / max(1, len(sorted_values)), 1)

aoi_peak_sorted = sorted(t[2] for t in tiles)
use_hours = list(range(USE_HOUR_START, USE_HOUR_END + 1))

records = []
for _, r in parks.iterrows():
    poly, hourly_c, peak_c, peak_h, mean_c = tile_for(r.latitude, r.longitude)
    use_window_temps = [hourly_c[h] for h in use_hours]
    hours_above_caution = sum(1 for v in use_window_temps if v >= HI_CAUTION_C)
    records.append({
        'peak_temp_c'                  : round(peak_c, 1),
        'peak_hour'                    : peak_h,
        'daily_mean_temp_c'            : round(mean_c, 1),
        'diurnal_swing_c'              : round(peak_c - min(hourly_c), 1),
        'aoi_percentile'               : _percentile_rank(peak_c, aoi_peak_sorted),
        'hours_above_caution_in_use'   : hours_above_caution,
        'use_window_peak_c'            : round(max(use_window_temps), 1),
    })
parks = pd.concat([parks.reset_index(drop=True), pd.DataFrame(records)], axis=1)
parks = parks.sort_values('peak_temp_c', ascending=False).reset_index(drop=True)
parks.insert(0, 'rank', parks.index + 1)

parks[['rank', 'park_id', 'name', 'type', 'acres',
       'peak_temp_c', 'peak_hour', 'daily_mean_temp_c',
       'aoi_percentile', 'hours_above_caution_in_use', 'use_window_peak_c']]

---
## Step 4 — Park-network overview map (M1)

### What you are doing
Drop the AOI heatmap (daily-average temperature, equal-interval classes) under your park markers. Marker size scales with peak temperature; color encodes park type so the parks director can see *which kinds of public spaces* are sitting in the heat — a regional park with high attendance vs. a small playground gets different attention.

### Why this matters (public health)
This is the briefing slide. Before any per-park diagnosis, the director sees the network in city context — the same way an incident commander sees an operational map. Type-color encoding catches systemic patterns: "all the playgrounds happen to be in the hot quarter of town" is a finding even before you look at any single park.

In [ ]:
N_BINS = len(TCM_COLORS)
tile_avgs = [t[4] for t in tiles]
least, highest = min(tile_avgs), max(tile_avgs)
interval = (highest - least) / N_BINS if highest > least else 0.0
class_entries = [
    {'min': least + i * interval,
     'max': least + (i + 1) * interval,
     'color': TCM_COLORS[i % len(TCM_COLORS)]}
    for i in range(N_BINS)
]
def _class_color(t):
    if t is None or interval == 0: return class_entries[0]['color']
    idx = int((t - least) / interval)
    return class_entries[max(0, min(idx, N_BINS - 1))]['color']

heatmap_features = [{
    'type': 'Feature',
    'geometry': mapping(poly),
    'properties': {
        'avg_str': f'{avg:.2f} °C',
        'fillColor': _class_color(avg),
    },
} for (poly, _, _, _, _), avg in zip(tiles, tile_avgs)]

min_peak = parks['peak_temp_c'].min()
center = [parks['latitude'].mean(), parks['longitude'].mean()]
m1 = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': heatmap_features},
    style_function=lambda f: {
        'fillColor': f['properties']['fillColor'],
        'color':     f['properties']['fillColor'],
        'weight':    0.6, 'fillOpacity': 0.75, 'opacity': 1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['avg_str'], aliases=['Avg temp'], sticky=True),
).add_to(m1)
for _, p in parks.iterrows():
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=4 + (p.peak_temp_c - min_peak) * 1.4,
        color='#000', weight=1.2,
        fill=True, fill_color=PARK_TYPE_PALETTE.get(p['type'], '#888'),
        fill_opacity=0.95,
        popup=(f"<b>#{int(p['rank'])} {p['name']}</b><br/>"
               f"{p['type']} · {p['acres']:.1f} acres<br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {int(p.peak_hour):02d}:00<br/>"
               f"hours ≥ NOAA Caution (10–18): {int(p['hours_above_caution_in_use'])}<br/>"
               f"AOI percentile: {p['aoi_percentile']}"),
    ).add_to(m1)

type_rows = ''.join(
    f'<tr><td style="background:{c};width:14px;height:14px;border:1px solid #000;border-radius:50%;"></td>'
    f'<td style="padding-left:8px;">{name}</td></tr>'
    for name, c in PARK_TYPE_PALETTE.items()
)
m1.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;'
    'padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    '<b>Park type</b>'
    f'<table>{type_rows}</table>'
    '<div style="color:#666;margin-top:6px;">marker size ∝ measured peak temp</div>'
    '</div>'
))
m1.fit_bounds([[aoi_bounds[1], aoi_bounds[0]], [aoi_bounds[3], aoi_bounds[2]]])
m1

---
## Step 5 — Above-median hot exposures (M2)

### What you are doing
Filter to parks whose peak temperature is at or above the network median, and render only the heatmap tiles those parks sit on. This isolates "the parks we have to talk about" from the rest.

### Why this matters (public health)
A parks director cannot diagnose 12 parks at the depth of satellite + street view + env-params. Step 5 picks the ones where the heat measurement already justifies a closer look — *before* spending a single API call on satellite or env-params. The cut is empirical (the median splits the network in half), not invented.

In [ ]:
median_peak = parks['peak_temp_c'].median()
hot = parks[parks['peak_temp_c'] >= median_peak].copy()

joined, seen = [], set()
for _, p in hot.iterrows():
    pt = Point(p.longitude, p.latitude)
    matched = next((t for t in tiles if t[0].contains(pt)), None)
    if matched is None:
        continue
    poly, _, peak_c, peak_h, _ = matched
    key = (round(poly.centroid.x, 6), round(poly.centroid.y, 6))
    if key in seen: continue
    seen.add(key)
    joined.append({'type':'Feature', 'geometry': mapping(poly),
                   'properties':{'peak_str': f'{peak_c:.2f} °C peak @ {peak_h:02d}:00',
                                 'fillColor': _class_color(peak_c)}})

m2 = folium.Map(location=center, zoom_start=13, tiles='cartodbpositron')
folium.GeoJson(
    {'type':'FeatureCollection','features':joined},
    style_function=lambda f: {
        'fillColor': f['properties']['fillColor'],
        'color':     '#000', 'weight': 1.5, 'fillOpacity': 0.9,
    },
    tooltip=folium.GeoJsonTooltip(fields=['peak_str'], aliases=['Tile peak'], sticky=True),
).add_to(m2)
for _, p in hot.iterrows():
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=8, color='black', weight=1.2,
        fill=True, fill_color=PARK_TYPE_PALETTE.get(p['type'], '#888'), fill_opacity=0.95,
        popup=(f"<b>#{int(p['rank'])} {p['name']}</b><br/>"
               f"{p['type']} · {p['acres']:.1f} acres<br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {int(p.peak_hour):02d}:00<br/>"
               f"hours ≥ NOAA Caution (10–18): {int(p['hours_above_caution_in_use'])}"),
    ).add_to(m2)
if not hot.empty:
    m2.fit_bounds([[hot['latitude'].min() - 0.004, hot['longitude'].min() - 0.004],
                   [hot['latitude'].max() + 0.004, hot['longitude'].max() + 0.004]])

print(f'{len(hot)} parks at or above the network median peak ({median_peak:.1f}°C); '
      f'{len(joined)} unique heatmap tiles after the spatial join.')
display(m2)
hot[['rank','park_id','name','type','peak_temp_c','peak_hour','hours_above_caution_in_use','aoi_percentile']]

---
## Step 6 — Surface diagnosis (satellite segmentation, top-N)

### What you are doing
For the top-N hottest parks, call `client.satellite_segmentation` and characterize the surface mix: building, road, tree, grass, sidewalk. We bucket those into `canopy_pct` and `impervious_pct` for the recommendation logic, and chart the full breakdown.

### Why this matters (public health)
Temperature alone tells you *what*; the surface mix tells you *why*. A park hitting 36 °C with 8 % canopy is a tree-planting candidate (USDA i-Tree program). A park hitting 36 °C with 70 % impervious paving is a cool-pavement candidate (EPA Heat Island Reduction). The difference matters because the funding source, the timeline, and the operating-budget impact are different — and the parks director needs to file the right grant application.

In [ ]:
IMPERV_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings',
               'rooftop', 'rooftops', 'sidewalk', 'earth', 'bare', 'ground'}
CANOPY_KEYS = {'tree', 'trees', 'vegetation', 'greenery', 'park'}
GRASS_KEYS  = {'grass'}

def _bucket(segments, keys):
    total = 0.0
    for cls, pct in (segments or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

def _first_b64(value):
    if isinstance(value, list): return value[0] if value else None
    return value

top_n = parks.head(TOP_N_TO_ENRICH).copy()
seg_data = {}
sat_imgs = {}
sat_source = {}

if CACHED:
    # Single bundled satellite cache file. We use it to demonstrate the diagnosis at
    # the top park; the same call would run independently per park in live mode.
    with open(SATELLITE_JSON, 'r', encoding='utf-8') as f:
        sat_doc = json.load(f)
    seg_block = sat_doc.get('segmentation', {}) or {}
    cached_segments = seg_block.get('segments', {}) or {}
    cached_imgs = {'orig': _first_b64(sat_doc.get('orignal_image')),
                   'seg' : seg_block.get('image_content')}
    for _, r in top_n.iterrows():
        seg_data[r.park_id] = cached_segments
        sat_imgs[r.park_id] = cached_imgs
        sat_source[r.park_id] = 'cached sample'
    print(f'[cached] satellite sample applied to top {len(top_n)} parks for demo. '
          f'In live mode each park is fetched independently.')
elif HAVE_API:
    for _, r in top_n.iterrows():
        try:
            sat = client.satellite_segmentation(
                latitude=float(r.latitude), longitude=float(r.longitude),
                start_date=STUDY_DATE, start_time=STUDY_HOUR,
                filter_type=1, granularity=GRANULARITY_M, verbose=False,
            )
            res = (sat.get('result') or {})
            seg_block = res.get('segmentation', {}) or {}
            seg_data[r.park_id] = seg_block.get('segments', {}) or {}
            sat_imgs[r.park_id] = {'orig': _first_b64(res.get('orignal_image')),
                                   'seg' : seg_block.get('image_content')}
            sat_source[r.park_id] = 'live'
            print(f'  satellite #{int(r["rank"])} {r.park_id}: ok')
        except FortyGuardError as exc:
            print(f'⚠ satellite #{int(r["rank"])} {r.park_id} failed: {exc}')
else:
    print('[no API key] skipping satellite enrichment.')

top_n['canopy_pct']     = top_n['park_id'].map(
    lambda pid: _bucket(seg_data.get(pid), CANOPY_KEYS) if pid in seg_data else None)
top_n['impervious_pct'] = top_n['park_id'].map(
    lambda pid: _bucket(seg_data.get(pid), IMPERV_KEYS) if pid in seg_data else None)
top_n['grass_pct']      = top_n['park_id'].map(
    lambda pid: _bucket(seg_data.get(pid), GRASS_KEYS) if pid in seg_data else None)

# Stacked bar of every class for the top-N parks.
rows = [(pid, segs) for pid, segs in seg_data.items() if segs]
if rows:
    classes = sorted({c for _, s in rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.2, 0.7 * len(rows))))
    bottoms = [0.0] * len(rows)
    pids = [pid for pid, _ in rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of surrounding scene')
    ax.set_title('Surface composition at top-N parks')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

# Original + segmented imagery at the top park.
if top_n['park_id'].iloc[0] in sat_imgs:
    pid = top_n['park_id'].iloc[0]
    imgs = sat_imgs[pid]
    def _img_tag(b64, label):
        if not b64:
            return (f'<div style="padding:1em;border:1px dashed #888;width:280px;height:280px;'
                    f'display:flex;align-items:center;justify-content:center;font:11px sans-serif">'
                    f'{label}: missing</div>')
        return (f'<figure style="margin:0"><img src="data:image/png;base64,{b64}" '
                f'style="width:280px;border:1px solid #888;"/>'
                f'<figcaption style="text-align:center;font:11px sans-serif">{label}</figcaption></figure>')
    display(HTML(
        f'<div style="font:12px sans-serif;margin:6px 0;"><b>#1 {pid}</b> — {top_n.iloc[0]["name"]} '
        f'(source: {sat_source.get(pid,"?")})</div>'
        f'<div style="display:flex;gap:12px;flex-wrap:wrap;">'
        + _img_tag(imgs.get('orig'), 'original') + _img_tag(imgs.get('seg'), 'segmented')
        + '</div>'))

top_n[['rank','park_id','name','type','peak_temp_c','canopy_pct','impervious_pct','grass_pct']]

---
## Step 7 — Street-view ground truth on #1

### What you are doing
Pulling the front-facing street view at the highest-ranked park and computing class shares — sky, tree, building, road, sidewalk. The `sky_pct` is the key public-health number: a park entrance with high sky fraction and low tree fraction is a place where families and seniors stand, unshaded, while waiting.

### Why this matters (public health)
Satellite shows the canopy from above; the public stands at the curb. A park can have decent overhead canopy in the interior but still fail the people lining up at the entrance, the parents waiting at pickup, or the seniors arriving on foot. NRPA shade-equity guidance specifically calls out unshaded gathering points as a candidate for shade-structure capital programs.

In [ ]:
TREE_KEYS_SV     = {'tree', 'trees', 'vegetation', 'greenery', 'grass'}
BUILDING_KEYS_SV = {'building', 'buildings', 'wall'}
SKY_KEYS_SV      = {'sky'}
ROAD_KEYS_SV     = {'road', 'roads', 'pavement', 'sidewalk'}

def _share(segs, keys):
    total = 0.0
    for cls, pct in (segs or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

sv_segs = None
sv_imgs = None
sv_src  = None
rank1 = parks.iloc[0]

if CACHED:
    with open(STREETVIEW_JSON, 'r', encoding='utf-8') as f:
        sv_doc = json.load(f)
    front = sv_doc.get('front') or {}
    sv_segs = front.get('segments', {}) or {}
    sv_imgs = {'orig': front.get('original_image'),
               'seg' : front.get('segmented_image'),
               'date': front.get('image_date', 'n/a')}
    sv_src = 'cached sample'
elif HAVE_API:
    try:
        sv = client.street_view_segmentation(
            latitude=float(rank1.latitude), longitude=float(rank1.longitude),
            verbose=False,
        )
        front = ((sv.get('result') or {}).get('front') or {})
        sv_segs = front.get('segments', {}) or {}
        sv_imgs = {'orig': front.get('original_image'),
                   'seg' : front.get('segmented_image'),
                   'date': front.get('image_date', 'n/a')}
        sv_src = 'live'
    except FortyGuardError as exc:
        print(f'⚠ street-view call failed ({exc}); skipping ground-truth step.')
else:
    print('[no API key] skipping street-view ground truth.')

if sv_segs is not None:
    sv_tree = _share(sv_segs, TREE_KEYS_SV)
    sv_bldg = _share(sv_segs, BUILDING_KEYS_SV)
    sv_sky  = _share(sv_segs, SKY_KEYS_SV)
    sv_road = _share(sv_segs, ROAD_KEYS_SV)
    parks.loc[0, 'sv_tree_pct']      = sv_tree
    parks.loc[0, 'sv_building_pct']  = sv_bldg
    parks.loc[0, 'sv_sky_pct']       = sv_sky
    parks.loc[0, 'sv_road_pct']      = sv_road
    print(f'#1 {rank1.park_id} {rank1["name"]}: '
          f'tree {sv_tree}% · building {sv_bldg}% · sky {sv_sky}% · road/sidewalk {sv_road}%  '
          f'(source: {sv_src})')

    if sv_imgs and (sv_imgs.get('orig') or sv_imgs.get('seg')):
        def _img(b64, label):
            if not b64:
                return (f'<div style="padding:1em;border:1px dashed #888;width:340px;height:200px;'
                        f'display:flex;align-items:center;justify-content:center;font:11px sans-serif">'
                        f'{label}: missing</div>')
            return (f'<figure style="margin:0"><img src="data:image/jpeg;base64,{b64}" '
                    f'style="width:340px;border:1px solid #888;"/>'
                    f'<figcaption style="text-align:center;font:11px sans-serif">{label}</figcaption></figure>')
        display(HTML(
            f'<div style="font:12px sans-serif;margin:6px 0;">'
            f'<b>#{int(rank1["rank"])} {rank1.park_id} — {rank1["name"]}</b>  '
            f'(imagery {sv_imgs.get("date","n/a")})</div>'
            f'<div style="display:flex;gap:12px;flex-wrap:wrap;">'
            + _img(sv_imgs.get('orig'), 'original') + _img(sv_imgs.get('seg'), 'segmented')
            + '</div>'))

---
## Step 8 — Diurnal driver profile (env-params, top-N)

### What you are doing
Calling `client.environmental_parameters` at the top-N parks for the use window (10:00–18:00) and reading off heat index, apparent temperature, and relative humidity. We compute three direct measurements per park: peak heat index, peak hour, and hours-above-NOAA-Caution.

### Why this matters (public health)
Heat index is the variable NOAA's heat advisories are written in — not dry-bulb temperature. A park hitting 32 °C dry-bulb at 30 % RH is uncomfortable; a park hitting 30 °C dry-bulb at 70 % RH crosses the same NOAA Caution boundary because the body can't shed heat through sweat. The recommendation logic in the next step uses heat index (not dry-bulb) for any threshold that maps to a CDC BRACE or NOAA advisory action, exactly the way the published guidance is written.

In [ ]:
env_data   = {}
env_source = {}

if CACHED:
    with open(ENV_PARAMS_JSON, 'r', encoding='utf-8') as f:
        env_doc = json.load(f)
    locs = env_doc.get('locations') or []
    if locs:
        params = (locs[0] or {}).get('parameters', {}) or {}
        cached_env = {
            'heat_index_celsius'           : list(params.get('heat_index_celsius') or []),
            'apparent_temperature_celsius' : list(params.get('apparent_temperature_celsius') or []),
            'relative_humidity_percent'    : list(params.get('relative_humidity_percent') or []),
        }
        for _, r in top_n.iterrows():
            env_data[r.park_id] = cached_env
            env_source[r.park_id] = 'cached sample'
    print(f'[cached] env-params sample applied to top {len(top_n)} parks for demo.')
elif HAVE_API:
    for _, r in top_n.iterrows():
        try:
            env = client.environmental_parameters(
                latitude=float(r.latitude), longitude=float(r.longitude),
                temperature=float(r.peak_temp_c),
                start_date=STUDY_DATE,
                start_time=f'{USE_HOUR_START:02d}:00',
                end_time=f'{USE_HOUR_END:02d}:00',
                filter_type=2, verbose=False,
            )
            loc = (env.get('result') or {}).get('locations', [{}])[0]
            params = (loc or {}).get('parameters', {}) or {}
            env_data[r.park_id] = {
                'heat_index_celsius'           : list(params.get('heat_index_celsius') or []),
                'apparent_temperature_celsius' : list(params.get('apparent_temperature_celsius') or []),
                'relative_humidity_percent'    : list(params.get('relative_humidity_percent') or []),
            }
            env_source[r.park_id] = 'live'
            print(f'  env-params #{int(r["rank"])} {r.park_id}: ok')
        except FortyGuardError as exc:
            print(f'⚠ env-params #{int(r["rank"])} {r.park_id} failed: {exc}')
else:
    print('[no API key] skipping env-params enrichment.')

def _peak_metrics(series, full_day=True):
    """Return (peak_hi, peak_hour, hours_above_caution_in_use_window).
    full_day=True means hi is indexed 0..23 and we slice the use window;
    full_day=False means hi already covers only the use window."""
    hi = series.get('heat_index_celsius') or []
    if not hi:
        return None, None, None
    if full_day and len(hi) >= 24:
        use_slice = hi[USE_HOUR_START:USE_HOUR_END + 1]
        peak = max(use_slice)
        peak_h = USE_HOUR_START + use_slice.index(peak)
        above = sum(1 for v in use_slice if v is not None and v >= HI_CAUTION_C)
    else:
        peak = max(hi)
        peak_h = USE_HOUR_START + hi.index(peak)
        above = sum(1 for v in hi if v is not None and v >= HI_CAUTION_C)
    return round(peak, 1), peak_h, above

for col in ('peak_heat_index_c', 'peak_hi_hour', 'hi_hours_above_caution'):
    top_n[col] = None
for pid, s in env_data.items():
    full_day = len(s.get('heat_index_celsius') or []) >= 24
    peak, peak_h, above = _peak_metrics(s, full_day=full_day)
    mask = top_n['park_id'] == pid
    top_n.loc[mask, 'peak_heat_index_c']      = peak
    top_n.loc[mask, 'peak_hi_hour']           = peak_h
    top_n.loc[mask, 'hi_hours_above_caution'] = above

# Plot heat-index curves for the top parks.
for _, r in top_n.iterrows():
    s = env_data.get(r.park_id)
    if not s or not s.get('heat_index_celsius'):
        continue
    hi   = s['heat_index_celsius']
    appt = s.get('apparent_temperature_celsius') or []
    rh   = s.get('relative_humidity_percent') or []
    full_day = len(hi) >= 24
    hours = list(range(len(hi))) if full_day else list(range(USE_HOUR_START, USE_HOUR_START + len(hi)))

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(hours, hi, label='Heat index (°C)', color='#d62728', marker='o', markersize=3)
    if appt:
        ax.plot(hours[:len(appt)], appt, label='Apparent temp (°C)', color='#ff7f0e',
                marker='o', markersize=3)
    ax.axhline(HI_CAUTION_C, color='#fdae61', linestyle='--', alpha=0.7,
               label=f'NOAA Caution ({HI_CAUTION_C:.0f} °C)')
    ax.axhline(HI_EXTREME_CAUTION_C, color='#e34a33', linestyle='--', alpha=0.7,
               label=f'NOAA Extreme Caution ({HI_EXTREME_CAUTION_C:.0f} °C)')
    ax.axvspan(USE_HOUR_START - 0.5, USE_HOUR_END + 0.5,
               color='#bbbbbb', alpha=0.15, label='use window 10–18')
    if rh:
        ax_rh = ax.twinx()
        ax_rh.plot(hours[:len(rh)], rh, color='#1f77b4', alpha=0.45, label='RH (%)')
        ax_rh.set_ylabel('RH (%)', color='#1f77b4')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('°C')
    ax.set_title(f"Diurnal drivers — #{int(r['rank'])} {r.park_id} {r['name']}  ·  {env_source.get(r.park_id,'?')}")
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

top_n[['rank','park_id','name','peak_temp_c','peak_heat_index_c','peak_hi_hour','hi_hours_above_caution']]

---
## Step 9 — Action brief (declarative, threshold-triggered)

### What you are doing
For each top-N park, we evaluate every measurement against its published threshold. **No invented index.** A recommendation appears in the brief only when a real measurement crosses a real threshold — and the brief cites the program that funds the fix.

### Why this matters (public health)
This step is the difference between *"this park scored 0.74"* and *"the canopy here is 14 % vs. an EPA / USDA target of 25 %, so file an i-Tree planting plan"*. The first is something the parks director cannot defend in a council meeting; the second is something they can hand to the grants officer.

Triggers and citations:

| If the API measures | Crossing threshold | Recommendation | Program |
|---|---|---|---|
| `canopy_pct` | < 25 (EPA / USDA target) | Tree-planting plan | USDA Forest Service i-Tree · EPA Heat Island Reduction |
| `impervious_pct` | > 60 (EPA Heat Island guidance) | Cool-pavement retrofit | EPA Heat Island Reduction |
| `peak_heat_index_c` | ≥ 32 (NOAA Extreme Caution) | Posted heat advisory + activity-window guidance | CDC BRACE |
| `sv_sky_pct` | > 60 with `sv_tree_pct` < 15 | Shade structure at entrance / play area | NRPA Shade-Equity guidance |
| RH > 60 % with peak temp > 30 °C | (compound) | Splash-pad / mister feasibility | CDC heat-resilience programs |

In [ ]:
def _triggers_for(row):
    """Return list of (label, evidence_string, program) tuples — a recommendation per threshold crossed."""
    triggers = []
    canopy = row.get('canopy_pct')
    imperv = row.get('impervious_pct')
    hi_pk  = row.get('peak_heat_index_c')
    sky    = row.get('sv_sky_pct')
    tree_sv = row.get('sv_tree_pct')
    s_env  = env_data.get(row['park_id']) or {}
    rh_series = s_env.get('relative_humidity_percent') or []
    rh_peak = max(rh_series) if rh_series else None

    if canopy is not None and not pd.isna(canopy) and canopy < CANOPY_TARGET_PCT:
        triggers.append((
            'Tree-planting plan',
            f'satellite canopy {canopy:.1f}% vs. EPA / USDA target {CANOPY_TARGET_PCT:.0f}%',
            PROG_USDA_ITREE,
        ))
    if imperv is not None and not pd.isna(imperv) and imperv > IMPERVIOUS_TRIGGER:
        triggers.append((
            'Cool-pavement retrofit',
            f'satellite impervious {imperv:.1f}% > EPA guidance {IMPERVIOUS_TRIGGER:.0f}%',
            PROG_EPA_HEATISLAND,
        ))
    if hi_pk is not None and not pd.isna(hi_pk) and hi_pk >= HI_EXTREME_CAUTION_C:
        triggers.append((
            'Posted heat advisory + activity-window guidance',
            f'env-params peak heat index {hi_pk:.1f}°C ≥ NOAA Extreme Caution {HI_EXTREME_CAUTION_C:.0f}°C',
            PROG_CDC_BRACE,
        ))
    if (sky is not None and not pd.isna(sky) and sky > SKY_TRIGGER_PCT and
        tree_sv is not None and not pd.isna(tree_sv) and tree_sv < 15.0):
        triggers.append((
            'Shade structure at entrance / play area',
            f'street-view sky {sky:.1f}% with tree {tree_sv:.1f}% (NRPA shade-equity criterion)',
            PROG_NRPA_SHADE,
        ))
    if (rh_peak is not None and rh_peak > RH_HIGH_PCT and
        row.get('peak_temp_c') is not None and row.get('peak_temp_c') > 30.0):
        triggers.append((
            'Splash-pad / mister feasibility study',
            f'peak RH {rh_peak:.0f}% with peak temp {row["peak_temp_c"]:.1f}°C',
            PROG_CDC_HEAT_RES,
        ))
    return triggers

# Render an action brief per top-N park.
for _, r in top_n.iterrows():
    triggers = _triggers_for(r.to_dict() | {'park_id': r['park_id']})
    rank = int(r['rank'])
    header = (f'<div style="font:600 15px sans-serif;margin:14px 0 4px 0;color:#0d0d0d">'
              f'#{rank} · {r["park_id"]} — {r["name"]}'
              f'<span style="color:#555;font-weight:400"> · {r["type"]} · {r["acres"]:.1f} ac · '
              f'peak {r.peak_temp_c:.1f}°C @ {int(r.peak_hour):02d}:00</span></div>')
    if not triggers:
        body = ('<div style="padding:10px 14px;border-left:4px solid #1a9850;background:#f7fbf7;'
                'border-radius:0 4px 4px 0;color:#1a1a1a">'
                '<b>No published threshold crossed.</b> Annual monitoring sufficient.'
                '</div>')
    else:
        items = ''.join(
            f'<div style="padding:10px 14px;border-left:4px solid #d73027;background:#fff5f5;'
            f'border-radius:0 4px 4px 0;margin-bottom:8px;color:#1a1a1a">'
            f'<div style="font:600 13px sans-serif;color:#0d0d0d">→ {label}</div>'
            f'<div style="margin:4px 0;color:#333">Trigger: {evidence}.</div>'
            f'<div style="font-size:11px;color:#666;">Program: {program}</div>'
            f'</div>'
            for label, evidence, program in triggers
        )
        body = items
    display(HTML('<div style="border:1px solid #ccc;border-radius:6px;padding:14px 16px;margin:10px 0;'
                 'font:13px/1.5 -apple-system,sans-serif;background:#ffffff">'
                 + header + body + '</div>'))

---
## Step 10 — Per-park audit CSV + ranked priority map (M3)

### What you are doing
Compose the final per-park audit row: every column is a direct API output, and the recommendation cell is the concatenated set of trigger-driven recommendations (or "Annual monitoring" if nothing crossed). Save to CSV. Render a ranked map with markers sized by peak temperature and popups carrying the recommendation list.

### Why this matters (public health)
The CSV is what the parks office adds to the next budget cycle. The map is what the public-health director shows the city council. Both end at the same content — the difference is medium, not data.

In [ ]:
# Merge the top-N enriched columns back into the full park frame.
merge_cols = ['park_id', 'canopy_pct', 'impervious_pct', 'grass_pct',
              'peak_heat_index_c', 'peak_hi_hour', 'hi_hours_above_caution']
parks = parks.merge(top_n[merge_cols], on='park_id', how='left')

# For parks not in top-N, satellite/env-params columns are None — only heatmap-derived
# triggers will fire for them. That is the correct conservative behavior.
def _row_recommendation(row):
    triggers = _triggers_for(row)
    if not triggers:
        return 'Annual monitoring'
    return ' · '.join(f'{label} ({program.split(" · ")[0]})' for label, _, program in triggers)

parks['recommendation'] = parks.apply(
    lambda r: _row_recommendation(r.to_dict() | {'park_id': r['park_id']}), axis=1)

audit_cols = ['rank', 'park_id', 'name', 'type', 'acres',
              'peak_temp_c', 'peak_hour', 'daily_mean_temp_c', 'aoi_percentile',
              'hours_above_caution_in_use', 'use_window_peak_c',
              'canopy_pct', 'impervious_pct',
              'peak_heat_index_c', 'peak_hi_hour', 'hi_hours_above_caution',
              'sv_tree_pct', 'sv_sky_pct',
              'recommendation']
for c in audit_cols:
    if c not in parks.columns:
        parks[c] = None
audit = parks[audit_cols]
# CSV preserves the full audit (all parks); the on-screen map and table below show
# only the top-N that received the deeper satellite + street-view + env-params look.
audit.to_csv(OUTPUT_CSV, index=False)

n_recs = (parks['recommendation'] != 'Annual monitoring').sum()
print(f'Saved {len(audit)} parks to {OUTPUT_CSV}  (full audit covers every park)')
print(f'  {n_recs} of {len(parks)} parks have at least one threshold-triggered recommendation')
print(f'  Map and table below: top {TOP_N_TO_ENRICH} only')

# ── Priority tier per park, driven by the count of triggered recommendations ──
# 3+ triggers = highest priority, 2 = high, 1 = moderate, 0 = monitor.
# This connects the marker color directly to the declarative trigger logic in Step 9
# so the map legend names something the parks director can act on.
PRIORITY_TIERS = [
    ('Highest',  '#b30000', 14, 'three or more threshold-triggered recommendations'),
    ('High',     '#e34a33', 12, 'two threshold-triggered recommendations'),
    ('Moderate', '#fdae61', 10, 'one threshold-triggered recommendation'),
    ('Monitor',  '#1a9850',  6, 'no published threshold crossed'),
]
def _priority_for(rec_str):
    if rec_str == 'Annual monitoring':
        return PRIORITY_TIERS[3]                       # monitor
    n = rec_str.count(' · ') + 1
    if n >= 3: return PRIORITY_TIERS[0]                # highest
    if n == 2: return PRIORITY_TIERS[1]                # high
    return PRIORITY_TIERS[2]                            # moderate

top_audit = audit[audit['rank'] <= TOP_N_TO_ENRICH].copy()
top_audit['priority'] = top_audit['recommendation'].apply(lambda r: _priority_for(r)[0])

# ── M3 — final ranked map (top-N only) ────────────────────────
peak_lo = float(top_audit['peak_temp_c'].min())
peak_hi = float(top_audit['peak_temp_c'].max())
top_lats = top_audit.merge(parks[['park_id','latitude','longitude']], on='park_id')

m3_center = [top_lats['latitude'].mean(), top_lats['longitude'].mean()]
m3 = folium.Map(location=m3_center, zoom_start=12, tiles='cartodbpositron')
for _, p in top_lats.iterrows():
    priority_label, color, base_radius, _why = _priority_for(p['recommendation'])
    # Marker size carries peak-temperature (continuous signal) on top of the priority base size.
    if peak_hi > peak_lo:
        size_bump = (p['peak_temp_c'] - peak_lo) / (peak_hi - peak_lo) * 6
    else:
        size_bump = 0
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=base_radius + size_bump,
        color='black', weight=1.2,
        fill=True, fill_color=color, fill_opacity=0.92,
        tooltip=f"#{int(p['rank'])} {p['park_id']} — {priority_label} priority",
        popup=(f"<b>#{int(p['rank'])} {p['name']}</b><br/>"
               f"{p['type']} · {p['acres']:.1f} acres<br/>"
               f"<b>Priority: {priority_label}</b><br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {int(p.peak_hour):02d}:00<br/>"
               f"hours ≥ NOAA Caution (10–18): {int(p['hours_above_caution_in_use'])}<br/>"
               f"<b style='color:#a00;'>→ {p['recommendation']}</b>"),
    ).add_to(m3)

if len(top_lats):
    m3.fit_bounds([[top_lats['latitude'].min() - 0.005, top_lats['longitude'].min() - 0.005],
                   [top_lats['latitude'].max() + 0.005, top_lats['longitude'].max() + 0.005]])

# Legend — explains both color (priority tier) and marker size (peak temperature)
priority_rows = ''.join(
    f'<tr>'
    f'<td style="padding:2px 6px;"><span style="display:inline-block;width:14px;height:14px;'
    f'border-radius:50%;background:{color};border:1px solid #000;"></span></td>'
    f'<td style="padding:2px 8px;font-weight:600;">{label}</td>'
    f'<td style="padding:2px 6px;color:#555;">{why}</td>'
    f'</tr>'
    for label, color, _radius, why in PRIORITY_TIERS
)
m3.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;'
    'padding:10px 14px;border:1px solid #888;font:12px sans-serif;max-width:360px;">'
    '<b>Priority tier</b><br/>'
    '<span style="color:#666;">color = number of threshold-triggered recommendations from Step 9</span>'
    f'<table style="margin-top:6px;border-collapse:collapse;">{priority_rows}</table>'
    '<div style="margin-top:8px;color:#666;font-size:11px;">'
    '<b>Marker size</b> ∝ measured peak temperature (within the top-N range). '
    'Larger marker = hotter park within the same priority tier.'
    '</div>'
    f'<div style="margin-top:6px;color:#666;font-size:11px;">'
    f'Showing top {TOP_N_TO_ENRICH} of {len(parks)} parks (the ones that received the deep-dive in Steps 6–8). '
    f'Full audit in {OUTPUT_CSV.name}.'
    '</div>'
    '</div>'
))
display(m3)

# Display table — top-N only, with the priority column up front for at-a-glance reading.
display_cols = ['rank', 'park_id', 'name', 'type', 'priority',
                'peak_temp_c', 'peak_hour', 'hours_above_caution_in_use',
                'canopy_pct', 'impervious_pct', 'peak_heat_index_c',
                'sv_sky_pct', 'recommendation']
top_audit[display_cols]

---
## Wrap-up

Starting from a parks point CSV you now have:

| Artifact | Audience |
|----------|----------|
| Per-park measurement table (every column is a direct API output) | Parks-and-rec analytics |
| **M1** park-network overview map | Briefing slide 1 |
| **M2** above-median hot exposures map | Briefing slide 2 |
| **M3** final ranked priority map with recommendations in the popup | Council meeting |
| Surface-composition stacked bar (top-N) | Grant-writing pack |
| Diurnal heat-index curves (top-N) | Posted heat-advisory documentation |
| Per-park action brief HTML cards | Parks director — go/no-go for each grant application |
| `outputs/public_parks_heat_audit.csv` | Inclusion in next-budget-cycle capital request |

**No invented index.** Every column is a direct API output. **No dollar value.** Every recommendation is keyed to a published national program (NOAA Heat Index, EPA Heat Island Reduction, USDA Forest Service i-Tree, CDC BRACE, NRPA Shade-Equity, CDC heat-resilience). Set `CACHED=False` (with a `FORTYGUARD_API_KEY`) to rerun the same workflow over any park list in any city — only the input CSV needs to change.

**Apply this pattern to adjacent use cases**: outdoor public-school zones (recess + PE + drop-off shade), public-library outdoor courtyards, public-housing common areas, faith-based / community-center outdoor gathering spaces, public swimming-pool approaches and queues. The workflow — *public point list × diurnal heatmap × surface diagnosis × ground-truth × env-params → declarative trigger-recommendation table* — transfers directly to anywhere the public stands outside in the heat.